# Policy Simulation
## Layer 3 — PD·LGD·정책 변수를 대출 한도 정책 전환 및 손익 시뮬레이션

## 목적

이 노트북은 앞선 네 단계의 산출물을 하나로 결합합니다.

- `00_lgd_estimation.ipynb`: 데이터에서 LGD를 직접 추정할 수 없다는 구조적 한계를 확인하고, 바젤 III 기준 Optimistic 0.40 / Base 0.55 / Conservative 0.75 세 시나리오를 확정
- `01_data_mart.ipynb`: 4개 원천 테이블을 고객 단위로 통합하고, DTI(`AMT_ANNUITY/AMT_INCOME_TOTAL`)·CIR(`AMT_CREDIT/AMT_INCOME_TOTAL`)·LTV(`AMT_CREDIT/AMT_GOODS_PRICE`)를 파생 피처로 확정
- `02_eda_policy_design.ipynb`: 위 세 변수 각각이 부도율과 어떤 관계인지 검증 — **LTV는 1.2를 변곡점으로 하는 독립적 위험 신호**, **DTI는 단독 판별력이 약해 PD와 결합했을 때만 보조 조건**, **CIR은 한도 규모의 상한 기준**(저위험≤3.0배/중위험3.0~5.0배/고위험5.0~7.5배/한도초과 7.5배 초과, 실용 상한 약 12배)으로 역할이 확정됨
- `04_calibration.ipynb`: XGBoost 원시 확률(ECE 0.3113)을 Platt Scaling으로 보정해 ECE 0.0061까지 낮추면서 AUC 0.7689를 그대로 보존

이 노트북의 목적은 신용위험 예측 결과를 `EL = PD × LGD × EAD` 관점에서 해석하고, 이를 승인 및 한도 정책 매트릭스로 연결해 실제 대출 의사결정에 반영하는 과정을 정리하는 데 있습니다.

진행 순서는 다음과 같습니다.

1. PD와 원본 정책 변수(EAD, 소득, DTI/CIR/LTV) 결합
2. PD 구간(risk tier)을 포트폴리오 평균 부실률 대비 배수로 설계하고, 실제 부도율로 타당성 확인
3. 02에서 확정한 LTV·CIR·DTI의 역할대로 정책 매트릭스 설계 (임의 분위수가 아닌, 02에서 검증한 실제 임계값 사용)
4. 정책 적용 전(전수 승인 가정) 대비 적용 후 부실률·기대손실(EL)·승인 한도 변화 계산
5. LGD 3개 시나리오에 대한 민감도 비교
6. `06_monitoring.ipynb`에서 쓸 산출물 저장


## Step 1. 데이터 결합

`val_scored.csv`에는 고객 ID·TARGET·`PD_calibrated`만 있으므로, 정책 설계에 필요한 EAD·소득·DTI·CIR·LTV는 `val_raw.csv`(01_data_mart.ipynb에서 이미 계산된 값)에서 다시 가져와 결합합니다.

EAD(Exposure at Default)는 부도 시점의 익스포저 금액입니다. 이 데이터셋에는 실제 인출 잔액 이력이 없으므로, 신청/승인 금액인 `AMT_CREDIT`을 EAD의 proxy로 사용합니다(실제 utilization을 반영하지 못해서 EAD가 과대 또는 과소 추정될 수 있고, 그래서 EL과 한도 산정이 보수적으로 치우칠 수 있습니다.)

In [2]:
import numpy as np
import pandas as pd

val_raw    = pd.read_csv('../data/val_raw.csv')
val_scored = pd.read_csv('../data/val_scored.csv')  # 04_calibration.ipynb 산출물

id_col = 'SK_ID_CURR' if 'SK_ID_CURR' in val_raw.columns else val_raw.columns[0]
policy_df = val_raw.merge(val_scored[[id_col, 'PD_calibrated']], on=id_col, how='inner')
policy_df['EAD'] = policy_df['AMT_CREDIT']  # 신청/승인 금액을 EAD proxy로 사용

baseline_bad_rate = policy_df['TARGET'].mean()
print(f"정책 시뮬레이션 대상: {len(policy_df):,}명")
print(f"포트폴리오 평균 부실률(정책 적용 전, 전수 승인 가정): {baseline_bad_rate:.4f}")
print(f"  → 04_calibration.ipynb / 01_data_mart.ipynb에서 확인된 부도율(0.0807)과 일치 확인용")


정책 시뮬레이션 대상: 61,503명
포트폴리오 평균 부실률(정책 적용 전, 전수 승인 가정): 0.0807
  → 04_calibration.ipynb / 01_data_mart.ipynb에서 확인된 부도율(0.0807)과 일치 확인용


## Step 2. PD 구간(Risk Tier) 설계

PD 구간의 경계는 임의의 숫자가 아니라 **포트폴리오 평균 부실률 대비 실제 부도율이 몇 배인지**를 기준으로 설계합니다. 먼저 `PD_calibrated`를 10분위로 구간화해 모델의 위험 순위가 실제 부도율과 일관되게 대응하는지 확인하고, 포트폴리오 평균 부실률 대비 부도율 배수를 기준으로 Risk Tier 경계를 설계합니다.

In [3]:
policy_df['pd_decile'] = pd.qcut(policy_df['PD_calibrated'], 10, labels=False, duplicates='drop') + 1

decile_summary = policy_df.groupby('pd_decile').agg(
    n=('TARGET', 'size'),
    pd_mean=('PD_calibrated', 'mean'),
    actual_default_rate=('TARGET', 'mean'),
).reset_index()
decile_summary['vs_baseline'] = (decile_summary['actual_default_rate'] / baseline_bad_rate).round(2)

print(decile_summary.round(4).to_string(index=False))


 pd_decile    n  pd_mean  actual_default_rate  vs_baseline
         1 6151   0.0135               0.0086         0.11
         2 6150   0.0186               0.0167         0.21
         3 6150   0.0243               0.0268         0.33
         4 6150   0.0315               0.0363         0.45
         5 6151   0.0412               0.0439         0.54
         6 6150   0.0548               0.0618         0.77
         7 6150   0.0751               0.0777         0.96
         8 6150   0.1066               0.1075         1.33
         9 6150   0.1608               0.1428         1.77
        10 6151   0.2799               0.2852         3.53


decile 1~10에 걸쳐 `actual_default_rate`가 0.86% → 1.67% → 2.68% → 3.63% → 4.39% → 6.18% → 7.77% → 10.75% → 14.28% → 28.52%로 단조 증가했습니다. 이는 PD가 고객의 위험 순위를 일관되게 구분하고 있음을 보여주며, 이후 tier 경계를 설계할 때 PD를 기준 축으로 활용할 수 있는 근거가 됩니다.

`vs_baseline` 기준으로 Step 3의 tier 경계를 데이터에 대입해보면 다음과 같습니다.
- **0.5배 경계**(저위험/중위험): decile 4(0.45배)와 decile 5(0.54배) 사이에서 교차 — 하위 약 40%가 저위험 tier에 해당
- **2배 경계**(중위험/고위험): decile 9(1.77배)와 decile 10(3.53배) 사이에서 교차 — 고위험 tier는 사실상 최상위 10분위 안에서만 시작
- **4배 경계**(고위험/거절): 최상위 decile 10의 평균조차 3.53배로 4배에 도달하지 못함

따라서 PD 기반 tier는 저위험부터 고위험까지의 광범위한 위험 분리를 담당하고, 4배 이상과 같은 극단 구간은 decile 10 내부의 꼬리 영역에만 존재하므로 거절 판단은 CIR 12배 초과와 같은 절대 기준이 맡는 다층 구조로 해석할 수 있습니다.

## Step 3. 정책 매트릭스 설계

**1) PD 기반 기본 tier 및 tier별 한도 배수 최적화**

한도 정책의 핵심 질문은 두 가지입니다. **누구를 승인할 것인가**, 그리고 승인한다면 **얼마까지 빌려줄 것인가**. 앞의 질문은 PD로 설명할 수 있지만, 뒤의 질문은 PD만으로는 충분하지 않습니다다. 같은 저위험 고객이라도 한도를 3배로 주는 경우와 5배로 주는 경우는 부실률에 미치는 영향이 다르므로, 한도는 별도의 계산과 정책 매트릭스로 설계해야 합니다.

먼저 PD를 절대 확률값이 아니라 포트폴리오 평균 부실률 대비 배수로 구간화합니다. 절대값 기준(예: "PD 5% 이하")은 포트폴리오 전체 위험 수준이 변할 때마다 재조정이 필요하지만, 배수 기준은 상대적 위험도를 그대로 반영하기 때문에 정책 강도를 안정적으로 유지할 수 있습니다.

tier가 정해지면 다음은 각 tier에 얼마까지 빌려줘도 회사가 감당할 수 있는지를 정하는 단계입니다. 이 배수를 정하는 구체적인 기준과 방법은 아래 3)에서 다룹니다.

**2) 보조 변수 결합**

PD 하나에만 의존하지 않도록, 02에서 판별력과 정책적 역할이 다르다고 확인한 세 변수를 각각의 기능에 맞게 결합합니다. 세 변수를 모두 같은 방식으로 하향 조정하지 않는 이유는, 각 변수가 위험 분리, 부담 수준, 거절 판단에서 서로 다른 역할을 맡기 때문입니다.

- **LTV (독립적 위험 신호, 기준 1.2)**: LTV가 1.2를 넘으면 부도율이 8.0% → 11.6% → 13.6%로 뚜렷한 상승을 확인했습니다. PD 등급과 무관하게 한 단계 하향합니다.
- **CIR (한도 상한 기준)**: 02에서 저위험(≤3.0배)/중위험(3.0~5.0배)/고위험(5.0~7.5배)/한도초과(7.5배 초과)로 구관화 했습니다. 신청 시점 CIR이 이미 한도초과 구간이면 PD 등급과 무관하게 한 단계 하향 적용합니다. 또한 실용 상한으로 도출된 **약 12배**를 초과하는 경우에는, tier와 무관하게 거절 처리합니다.
- **DTI (PD와 결합했을 때만 보조 조건)**: 02에서 DTI 구간별 부도율은 7.2%에서 8.8% 수준으로 큰 차이를 보이지 않아 단독 판별력은 제한적이었습니다. 따라서 DTI는 저위험 고객에는 적용하지 않고, PD가 이미 중위험 이상인 고객에 한해 DTI 40% 초과 시 추가로 한 단계 하향하는 보조 규칙으로 사용합니다.

하향 조건은 중복 적용 가능하며(예: LTV와 CIR 모두 해당 시 두 단계 하향), 최종 tier는 4단계(저위험/중위험/고위험/거절)를 넘어가지 않도록 거절에서 고정합니다.

In [4]:
TIER_ORDER = ['저위험', '중위험', '고위험', '거절']
TIER_IDX = {t: i for i, t in enumerate(TIER_ORDER)}

policy_df['pd_ratio_to_baseline'] = policy_df['PD_calibrated'] / baseline_bad_rate

def pd_base_tier(ratio):
    if ratio <= 0.5:
        return '저위험'
    elif ratio <= 2.0:
        return '중위험'
    elif ratio <= 4.0:
        return '고위험'
    else:
        return '거절'

policy_df['base_tier'] = policy_df['pd_ratio_to_baseline'].apply(pd_base_tier)

# LTV 1.2 이상 → 독립적 위험 신호
flag_ltv = policy_df['LTV'] > 1.2
# CIR 7.5배 초과 → 한도초과 구간
flag_cir_over = policy_df['CIR'] > 7.5
# DTI는 PD가 이미 중위험/고위험인 경우에만 결합 적용 (상한 40%)
elevated_pd = policy_df['base_tier'].isin(['중위험', '고위험'])
flag_dti = elevated_pd & (policy_df['DTI'] > 0.40)

downgrade_steps = flag_ltv.astype(int) + flag_cir_over.astype(int) + flag_dti.astype(int)
base_pos = policy_df['base_tier'].map(TIER_IDX)
final_pos = np.minimum(base_pos + downgrade_steps, len(TIER_ORDER) - 1)
policy_df['risk_tier'] = final_pos.map(lambda i: TIER_ORDER[i])

# CIR 실용 상한(약 12배) 초과 시 tier와 무관하게 절대 거절
policy_df.loc[policy_df['CIR'] > 12, 'risk_tier'] = '거절'

print(f"LTV>1.2 하향 대상: {flag_ltv.sum():,}건")
print(f"CIR>7.5 하향 대상: {flag_cir_over.sum():,}건")
print(f"DTI>0.40 하향 대상(중/고위험 한정): {flag_dti.sum():,}건")
print(f"CIR>12 절대 거절 대상: {(policy_df['CIR'] > 12).sum():,}건")

LTV>1.2 하향 대상: 12,978건
CIR>7.5 하향 대상: 5,969건
DTI>0.40 하향 대상(중/고위험 한정): 817건
CIR>12 절대 거절 대상: 875건


**3) tier별 한도 배수 결정**

tier별 한도 배수는 PD tier만으로 정해지지 않습니다. 각 tier가 승인하는 총 한도 대비 기대손실(EL)이 일정 비율을 넘지 않는 범위에서 역산해, 정책 강도와 손실 허용범위를 맞췄습니다. 즉, EL_ratio 상한은 해당 tier에서 허용 가능한 한도 개방 강도를 결정하는 기준입니다.

이 역산은 LGD 3개 시나리오 각각에 대해 수행합니다. LGD가 커질수록 같은 배수에서도 기대손실이 증가하므로, 시나리오별로 감내 가능한 한도 배수가 달라집니다. 최종적으로는 가장 보수적인 Conservative 시나리오에서도 EL_ratio 기준을 넘지 않는 tier만 승인 가능하다고 판정하고, 통과한 tier에는 `02_eda_policy_design.ipynb`에서 실제로 관측된 CIR 위험 구간 상한(저위험 3.0배/중위험 5.0배/고위험 7.5배)을 배수로 적용합니다. 안전 여부는 손실 기준으로 판단하고, 실제 승인 규모는 관측된 신청 패턴으로 정해 두 근거의 역할을 분리했습니다.

이 EL_ratio 상한은 데이터셋에서 자동으로 도출되는 값이 아니라 회사의 손실 감내 수준에 대한 정책적 판단이 필요하므로, 보수적/기본/완화 3개 가정으로 나누어 판정 결과의 민감도를 함께 확인했습니다. 이렇게 결정된 tier별 배수는 정책 적용 전후의 승인 집단 부실률, EL, 총 승인 한도 변화로 이어지며, 최종 정책의 타당성을 검증하는 기준이 됩니다.

In [5]:
LGD_SCENARIOS = {'Optimistic': 0.40, 'Base': 0.55, 'Conservative': 0.75}

# tier별 EL_ratio 상한 — 3개 가정 세트 (범위 2~12%대 내에서, tier 위험도에 비례해 배정)
THRESHOLD_SCENARIOS = {
    '보수적': {'저위험': 0.02, '중위험': 0.04, '고위험': 0.06},
    '기본':   {'저위험': 0.03, '중위험': 0.06, '고위험': 0.09},
    '완화':   {'저위험': 0.04, '중위험': 0.08, '고위험': 0.12},
}

CIR_TIER_CAP = {'저위험': 3.0, '중위험': 5.0, '고위험': 7.5}

tier_avg_pd = policy_df.groupby('risk_tier')['PD_calibrated'].mean()

# threshold 시나리오 x LGD 시나리오별 pass/fail 판정
pass_fail = {}
for scenario_name, thresholds in THRESHOLD_SCENARIOS.items():
    pass_fail[scenario_name] = {}
    for lgd_name, lgd in LGD_SCENARIOS.items():
        pass_fail[scenario_name][lgd_name] = {}
        for tier in ['저위험', '중위험', '고위험']:
            approx_el_ratio = tier_avg_pd[tier] * lgd
            pass_fail[scenario_name][lgd_name][tier] = approx_el_ratio <= thresholds[tier]

final_pass = {}
for scenario_name in THRESHOLD_SCENARIOS:
    final_pass[scenario_name] = {}
    for tier in ['저위험', '중위험', '고위험']:
        final_pass[scenario_name][tier] = all(
            pass_fail[scenario_name][lgd_name][tier] for lgd_name in LGD_SCENARIOS
        )

pass_summary = pd.DataFrame(final_pass).T[['저위험', '중위험', '고위험']]
print("threshold 시나리오별 tier 통과 여부(세 LGD 시나리오 모두 통과해야 True):")
print(pass_summary)

threshold 시나리오별 tier 통과 여부(세 LGD 시나리오 모두 통과해야 True):
      저위험    중위험    고위험
보수적  True  False  False
기본   True   True  False
완화   True   True   True


In [6]:
# 이후 계산은 '기본' 시나리오를 채택 — 보수적/완화는 민감도 확인용
CHOSEN_SCENARIO = '기본'

TIER_MULTIPLIER = {
    tier: (CIR_TIER_CAP[tier] if final_pass[CHOSEN_SCENARIO][tier] else 0.0)
    for tier in ['저위험', '중위험', '고위험']
}
TIER_MULTIPLIER['거절'] = 0.0

policy_df['income_multiplier'] = policy_df['risk_tier'].map(TIER_MULTIPLIER)
policy_df['proposed_limit'] = np.minimum(
    policy_df['AMT_INCOME_TOTAL'] * policy_df['income_multiplier'],
    policy_df['AMT_CREDIT']
)
policy_df.loc[policy_df['risk_tier'] == '거절', 'proposed_limit'] = 0
policy_df['approved'] = policy_df['proposed_limit'] > 0

tier_summary = policy_df.groupby('risk_tier').agg(
    n=('TARGET', 'size'),
    actual_default_rate=('TARGET', 'mean'),
    income_multiplier=('income_multiplier', 'first'),
    avg_limit=('proposed_limit', 'mean'),
).reindex(TIER_ORDER).round(4)
tier_summary['비중'] = (tier_summary['n'] / len(policy_df)).round(4)
print(f"채택 시나리오: {CHOSEN_SCENARIO}")
print(f"적용 배수: {TIER_MULTIPLIER}")
print(tier_summary.to_string())

채택 시나리오: 기본
적용 배수: {'저위험': 3.0, '중위험': 5.0, '고위험': 0.0, '거절': 0.0}
               n  actual_default_rate  income_multiplier    avg_limit      비중
risk_tier                                                                    
저위험        21200               0.0215                3.0  435296.8677  0.3447
중위험        22397               0.0694                5.0  522539.6079  0.3642
고위험        11997               0.1314                0.0       0.0000  0.1951
거절          5909               0.2332                0.0       0.0000  0.0961


**실험 결과**

**문제 정의**: tier별 한도 배수는 이 노트북에서 정책적으로 결정해야 하는 변수이며, 각 tier에 승인하는 총 한도 대비 기대손실(EL_ratio)이 회사의 손실 감내 기준을 넘지 않는지 확인해야 최종 확정할 수 있습니다.

**검증 설계**: 이 기준을 세 개의 LGD 가정(Optimistic 0.40 / Base 0.55 / Conservative 0.75) 각각에 대해 역산하고, 가장 회수 여건이 나쁜 Conservative 시나리오를 최종 통과 여부를 가르는 게이트로 사용했습니다. Base나 Optimistic 기준으로만 판정하면 실제 회수 여건이 예상보다 나쁠 때 정책이 감내하기 어려운 손실을 초래할 수 있기 때문입니다.

**결과**: 저위험 tier는 세 시나리오 모두에서 통과했으며, 고위험 tier는 완화(12%) 시나리오에서만 통과했습니다. 기본 시나리오(threshold 3%/6%/9%)에서는 저위험과 중위험만 통과하고 고위험은 탈락했습니다. 고위험 tier의 평균 PD(약 13.65%)에 Conservative LGD(0.75)를 곱한 손실률은 약 10.24%로 기본 시나리오 threshold(9%)를 약 1.24%p 초과해 탈락했고, Optimistic(5.46%)·Base(7.51%) 기준으로는 통과선 안에 들었지만 "세 시나리오 모두 통과해야 최종 승인"이라는 게이트 기준에는 미달했습니다. 반면 중위험은 Conservative 기준에서도 약 5.09%(PD 6.78% × LGD 0.75)로 threshold(6%)보다 0.91%p 낮아 안정적으로 통과했습니다.

**채택값**: 세 시나리오 게이트를 통과한 tier에 한해 실제로 관측된 CIR 위험 구간 상한을 기준으로 정했습니다. 저위험과 중위험은 각각 3.0배, 5.0배를 채택했고, 고위험은 세 시나리오를 모두 통과하지 못해 배수 0, 즉 거절로 처리했습니다. 이 값은 실제 신청 패턴에서 검증된 한도 상한이라는 점에서 정책 근거를 가집니다.

이 threshold(2~12% 범위의 보수적/기본/완화 가정)는 회사의 손실 감내 수준에 대한 내부 정책 결정값이므로, 위 3개 시나리오는 "정답"을 제시하기보다 가정에 따라 승인/거절 경계가 어디서 갈리는지(중위험은 4~6% 사이, 고위험은 9~12% 사이)를 보여주는 민감도 분석입니다. 실제 운영 전환 시에는 리스크위원회가 이 경계값을 확정해야 합니다.

이렇게 확정된 배수(저위험 3.0배/중위험 5.0배/고위험 거절)가 실제로 승인 집단의 부실률과 기대손실, 총 승인 한도를 어떻게 바꾸는지는 이어지는 Step 4에서 정책 적용 전후 비교로 확인합니다.

## Step 4. 정책 적용 전후 비교

정책 적용 전(before)은 PD와 무관하게 신청 금액(`AMT_CREDIT`) 전액을 승인하는 전수 승인 가정이고, 정책 적용 후(after)는 위에서 설계한 tier별 한도(`proposed_limit`, 거절 포함)를 승인하는 상황을 가정합니다. 두 상황의 차이가 곧 이 정책이 만드는 효과입니다.

비교 지표는 세 가지입니다.

1. **승인 집단 부실률**: 거절이 실제로 고위험 고객을 걸러내는지
2. **기대손실(EL)**: `PD_calibrated × LGD × EAD`를 승인된 익스포저에 대해 합산
3. **총 승인 한도**: 저위험 고객 한도 확대와 고위험 고객 축소/거절이 상쇄되어 전체 한도 규모가 어떻게 바뀌는지

In [7]:
def compute_el(df, exposure_col, lgd):
    return (df['PD_calibrated'] * lgd * df[exposure_col]).sum()

approved_df = policy_df[policy_df['approved']]

el_rows = []
for name, lgd in LGD_SCENARIOS.items():
    el_before = compute_el(policy_df, 'AMT_CREDIT', lgd)
    el_after  = compute_el(approved_df, 'proposed_limit', lgd)
    el_rows.append({
        '시나리오': name, 'LGD': lgd,
        'EL_before': el_before, 'EL_after': el_after,
        'EL_감소액': el_before - el_after,
        'EL_감소율': (el_before - el_after) / el_before,
    })
el_summary = pd.DataFrame(el_rows)
print(el_summary.round(0).to_string(index=False))

approval_rate      = policy_df['approved'].mean()
approved_bad_rate  = approved_df['TARGET'].mean()
limit_before_total = policy_df['AMT_CREDIT'].sum()
limit_after_total  = approved_df['proposed_limit'].sum()

print()
print(f"승인율: {approval_rate:.4f}")
print(f"승인 집단 부실률: {approved_bad_rate:.4f}  (정책 적용 전 포트폴리오 평균: {baseline_bad_rate:.4f})")
print(f"총 승인 한도: {limit_before_total:,.0f} → {limit_after_total:,.0f} "
      f"({(limit_after_total / limit_before_total - 1) * 100:+.1f}%)")


        시나리오  LGD    EL_before    EL_after       EL_감소액  EL_감소율
  Optimistic  0.0 1099451419.0 394283373.0  705168046.0     1.0
        Base  1.0 1511745701.0 542139638.0  969606063.0     1.0
Conservative  1.0 2061471410.0 739281325.0 1322190086.0     1.0

승인율: 0.7089
승인 집단 부실률: 0.0461  (정책 적용 전 포트폴리오 평균: 0.0807)
총 승인 한도: 36,751,982,445 → 20,931,613,194 (-43.0%)


**결과**: Base 시나리오(LGD 0.55) 기준으로 EL이 15.1억 → 5.4억으로 64.15% 감소했습니다. 이 감소율은 Optimistic(64.14%)·Conservative(64.14%)에서도 사실상 동일하게 나타났는데, 정책 효과가 특정 LGD 가정에만 의존하지 않는 구조적 결과임을 보여줍니다. `EL = PD × LGD × 한도`에서 LGD는 정책 적용 전후 양쪽에 동일하게 곱해지는 상수이므로, 감소율 계산에서는 상쇄됩니다. 즉 이 64% 감소는 LGD 가정과 무관하게 유지되는, **승인 및 한도 조정 정책 자체의 효과**로 볼 수 있습니다.

승인 집단 부실률은 4.61%로, 정책 적용 전 포트폴리오 평균(8.07%)의 약 절반 수준으로 낮아졌습니다. 이는 고위험 tier를 거절 처리한 것이 실제로 손실 가능성이 큰 고객을 걸러내고 있음을 시사하며, 앞서 채택한 배수(저위험 3.0배/중위험 5.0배/고위험 거절)가 위험 분리 목적에 부합함을 보여줍니다.

승인율은 70.89%로, tier_summary에서 확인한 저위험(34.47%)·중위험(36.42%) 비중의 합과 정확히 일치합니다. 총 승인 한도는 367.5억 → 209.3억으로 43.0% 줄었는데, 이는 두 방향의 상쇄 결과입니다 — 저위험 고객은 CIR 상한(3.0배)까지 확대되지만, 고위험(19.51%)·거절(9.61%) 구간, 즉 전체의 약 29.1%가 한도 0으로 전환되면서 축소 효과가 더 크게 작용했기 때문입니다.

**정책적 의미**: 이 정책은 총 대출 규모를 43% 줄이는 대신 EL을 64% 이상 감소시킨 보수적 트레이드오프를 만듭니다. 손실 감소폭이 한도 축소폭보다 크다는 점(EL -64% vs 한도 -43%)은, 한도 축소가 무작위가 아니라 실제로 위험이 높은 고객군에 집중적으로 작동했음을 시사합니다. 다만 한도 축소가 회사 수익, 특히 이자 수익에 미치는 영향은 이 노트북에서 다루지 않았으므로, 이 트레이드오프가 최종적으로 바람직한지는 수익성 지표까지 포함해 판단해야 합니다.

## Step 5. 시나리오 민감도 요약

LGD는 이 데이터에서 직접 추정할 수 없어 바젤 III 기준 3개 시나리오로 다뤘습니다(`00_lgd_estimation.ipynb`). 따라서 정책 효과를 하나의 숫자로 제시하기보다, Optimistic~Conservative 구간의 범위로 제시하는 것이 더 정직한 커뮤니케이션입니다. Base(0.55)를 대표값으로 쓰되, 이력서·보고서에는 항상 이 구간을 함께 표기합니다.


In [8]:
sensitivity = el_summary[['시나리오', 'LGD', 'EL_감소율']].copy()
sensitivity['EL_감소율'] = (sensitivity['EL_감소율'] * 100).round(2).astype(str) + '%'
print(sensitivity.to_string(index=False))

print(f"\nEL 감소율 범위: {el_summary['EL_감소율'].min()*100:.1f}% ~ {el_summary['EL_감소율'].max()*100:.1f}%"
      f"  (Base 시나리오: {el_summary.loc[el_summary['시나리오']=='Base', 'EL_감소율'].values[0]*100:.1f}%)")


        시나리오  LGD EL_감소율
  Optimistic 0.40 64.14%
        Base 0.55 64.14%
Conservative 0.75 64.14%

EL 감소율 범위: 64.1% ~ 64.1%  (Base 시나리오: 64.1%)


## Step 6. 산출물 저장

`06_monitoring.ipynb`는 이 정책을 실제로 적용했다고 가정했을 때 운영 중 분포·성능이 어떻게 흔들리는지를 다룹니다. 이를 위해 정책이 적용된 고객 단위 데이터(tier, 한도, 승인 여부)와 시나리오별 EL 요약을 저장합니다.


In [9]:
policy_output_cols = [id_col, 'TARGET', 'PD_calibrated', 'risk_tier', 'income_multiplier',
                       'proposed_limit', 'approved', 'DTI', 'CIR', 'LTV', 'AMT_CREDIT', 'AMT_INCOME_TOTAL']
policy_df[policy_output_cols].to_csv('../data/policy_simulation.csv', index=False)
el_summary.to_csv('../data/el_scenario_summary.csv', index=False)

print(f"저장 완료: ../data/policy_simulation.csv ({len(policy_df):,}행)")
print(f"저장 완료: ../data/el_scenario_summary.csv ({len(el_summary)}행)")


저장 완료: ../data/policy_simulation.csv (61,503행)
저장 완료: ../data/el_scenario_summary.csv (3행)


## 다음 단계에서는

이 노트북에서 확정한 것은 다음과 같습니다.

- **PD 기반 tier**: 절대 확률이 아닌 포트폴리오 평균 부실률 대비 배수(0.5배/2배/4배)로 정의
- **보조 변수 결합 방식**: 02_eda_policy_design.ipynb에서 검증된 역할대로 차등 적용 — LTV(1.2 초과, 독립 신호)와 CIR(7.5배 초과, 한도초과 구간)은 PD 등급과 무관하게 하향, DTI(40% 초과)는 PD가 이미 중위험 이상인 경우에만 결합 적용, CIR 12배 초과는 tier 무관 절대 거절
- **한도 정책**: 저위험 소득 5배 / 중위험 3배 / 고위험 1.5배 / 거절
- **효과 검증 방식**: EL·부실률·승인 한도를 정책 적용 전후로 비교하고, LGD 3개 시나리오에 대한 민감도를 함께 제시

다음 `06_monitoring.ipynb`에서는 이 정책이 실제 운영에 들어갔다고 가정하고, `policy_simulation.csv`의 승인 집단을 기준으로 입력 분포(PSI)와 PD 점수 분포, 지연 라벨 기반 성능(AUC/KS/Calibration) 추적 체계를 설계합니다.
